In [ ]:
import urllib.request
import os

model_path = 'hand_landmarker.task'
if not os.path.exists(model_path):
    print("모델 파일 다운로드 중...")
    url = "!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, model_path)
    print("다운로드 완료!")

In [ ]:
import cv2
import mediapipe as mp
import time

# ==============================================================================
# 1. MediaPipe Tasks API 핵심 클래스 로드
# ==============================================================================
# BaseOptions: 모델 파일 경로(.task) 및 디바이스(CPU/GPU) 설정 담당
BaseOptions = mp.tasks.BaseOptions

# HandLandmarker: 실제 손 관절 위치를 추론하는 메인 클래스
HandLandmarker = mp.tasks.vision.HandLandmarker

# HandLandmarkerOptions: 신뢰도 임계값, 인식 모드, 최대 손 개수 등을 세팅하는 옵션 클래스
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

# VisionRunningMode: 입력 데이터 형태 정의 (IMAGE: 정지영상, VIDEO: 비디오 파일, LIVE_STREAM: 실시간 웹캠)
VisionRunningMode = mp.tasks.vision.RunningMode


# ==============================================================================
# 2. 손 관절(Landmark) 연결선 정의 (시각화용)
# ==============================================================================
# 21개 관절 점 인덱스를 기반으로 손가락 마디마디를 잇는 튜플 리스트
HAND_CONNECTIONS = [
    # 엄지손가락 (Wrist 0 -> 손목 base 1 -> 관절 2, 3 -> 손끝 4)
    (0, 1), (1, 2), (2, 3), (3, 4),
    
    # 검지손가락 (Wrist 0 -> 검지 뿌리 5 -> 관절 6, 7 -> 손끝 8)
    (0, 5), (5, 6), (6, 7), (7, 8),
    
    # 중지손가락 (검지 뿌리 5 -> 중지 뿌리 9 -> 관절 10, 11 -> 손끝 12)
    (5, 9), (9, 10), (10, 11), (11, 12),
    
    # 약지손가락 (중지 뿌리 9 -> 약지 뿌리 13 -> 관절 14, 15 -> 손끝 16)
    (9, 13), (13, 14), (14, 15), (15, 16),
    
    # 새끼손가락 (약지 뿌리 13 -> 새끼 뿌리 17 -> 관절 18, 19 -> 손끝 20 + 손목 0 연결)
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20)
]


# ==============================================================================
# 3. HandLandmarker 옵션 설정 및 모델 로드
# ==============================================================================
options = HandLandmarkerOptions(
    # 다운로드받은 모델 파일(.task)의 상대/절대 경로 지정
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    
    # 비디오 스트림 처리 모드로 설정 (동적 타임스탬프와 함께 연속 프레임 추적)
    running_mode=VisionRunningMode.VIDEO,
    
    # 화면 내 감지할 최대 손 개수 설정 (1로 설정 시 가장 명확한 손 1개만 추적)
    num_hands=1,
    
    # 손 탐지(Detection) 최소 신뢰도 (0.5 = 50% 이상 확신할 때 손으로 인식)
    min_hand_detection_confidence=0.5,
    
    # 손 추적(Tracking) 최소 신뢰도 (프레임 간 손의 위치를 계속 추적하는 임계값)
    min_tracking_confidence=0.5
)

# 기본 연결된 웹캠 장치 열기 (0: 기본 카메라)
cap = cv2.VideoCapture(0)


# ==============================================================================
# 4. 실시간 추론 및 시각화 루프
# ==============================================================================
# 'with' 문을 통해 Landmarker 객체를 안전하게 생성 (작업 종료 시 메모리 자원 자동 해제)
with HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        # 웹캠으로부터 1개 프레임 읽어오기 (ret: 성공여부 Boolean, frame: BGR 이미지 배열)
        ret, frame = cap.read()
        if not ret:
            print("카메라 프레임을 읽어올 수 없습니다.")
            break

        # 거울 모드 처리를 위한 좌우 반전 (1: 좌우 반전)
        frame = cv2.flip(frame, 1)
        
        # OpenCV의 BGR 색상 채널을 MediaPipe가 지원하는 RGB 채널로 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # NumPy 배열 이미지를 MediaPipe의 전용 Image 객체로 래핑
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # Tasks API VIDEO 모드 필수값: 밀리초(ms) 단위의 단조 증가 타임스탬프 생성
        frame_timestamp_ms = int(time.time() * 1000)
        
        # RGB 이미지와 타임스탬프를 모델에 전달하여 3D 관절 추론 실행
        result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        # 원본 프레임의 높이(h), 너비(w) 추출 (정규화 좌표 -> 픽셀 좌표 변환용)
        h, w, _ = frame.shape
        
        # 손이 화면에서 성공적으로 감지되었는지 확인
        if result.hand_landmarks:
            # 감지된 손 개수만큼 반복 (num_hands=1 설정으로 최대 1회 실행)
            for hand_landmarks in result.hand_landmarks:
                pixel_coords = [] # 21개 관절의 (x, y) 픽셀 좌표를 담을 리스트
                
                # --------------------------------------------------------------
                # 4-1. 관절 점(Landmark) 위치 계산 및 출력
                # --------------------------------------------------------------
                for lm in hand_landmarks:
                    # lm.x, lm.y는 0.0~1.0 사이로 정규화된 값이므로 해상도(w, h)를 곱해 픽셀 단위로 변환
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    pixel_coords.append((cx, cy))
                    
                    # 관절 위치에 녹색 원 그리기 (원 중심: cx,cy / 반지름: 4px / 색상: BGR (0,255,0) / 채우기: -1)
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

                # --------------------------------------------------------------
                # 4-2. 관절 마디 연결선 그리기
                # --------------------------------------------------------------
                for start_idx, end_idx in HAND_CONNECTIONS:
                    # HAND_CONNECTIONS 상의 두 관절 픽셀 좌표 획득
                    p1 = pixel_coords[start_idx]
                    p2 = pixel_coords[end_idx]
                    
                    # 두 점 사이에 파란색 직선 그리기 (색상: BGR (255,0,0) / 두께: 2px)
                    cv2.line(frame, p1, p2, (255, 0, 0), 2)

        # 처리 결과 프레임을 "Hand Recognition" 윈도우 창에 표시
        cv2.imshow("Hand Recognition", frame)

        # 5밀리초 동안 키 입력을 대기하고 8비트 마스크 적용
        key = cv2.waitKey(5) & 0xFF
        # ESC 키(ASCII 27) 또는 'q' 키를 누르면 루프 종료
        if key == 27 or key == ord('q'):
            break

# ==============================================================================
# 5. 자원 해제
# ==============================================================================
# 웹캠 장치 점유 해제
cap.release()

# 오픈되어 있는 모든 OpenCV 그래픽 창 닫기
cv2.destroyAllWindows()

In [ ]:
# OpenCV 라이브러리: 비디오 프레임 처리 및 화면 출력을 담당
import cv2

# MediaPipe 라이브러리: 손 관절(Landmark) 추론 및 컴퓨터 비전 기능 제공
import mediapipe as mp

# random 라이브러리: 컴퓨터의 가위/바위/보 무작위 선택에 사용
import random

# time 라이브러리: MediaPipe Tasks API 입력에 필요한 밀리초 타임스탬프 계산
import time

# ==============================================================================
# 1. MediaPipe Tasks API 필수 모듈 및 설정 로드
# ==============================================================================
# BaseOptions: 모델 파일 경로(.task) 및 CPU/GPU 디바이스 지정 옵션
BaseOptions = mp.tasks.BaseOptions

# HandLandmarker: 21개 손 관절 좌표를 추론하는 메인 클래스
HandLandmarker = mp.tasks.vision.HandLandmarker

# HandLandmarkerOptions: 추론 신뢰도, 감지할 손 개수, 실행 모드를 세팅하는 클래스
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

# VisionRunningMode: 입력 형태 설정 (IMAGE: 이미지, VIDEO: 비디오 파일, LIVE_STREAM: 비동기 스트림)
VisionRunningMode = mp.tasks.vision.RunningMode

# 손 관절 뼈대를 이어줄 21개 랜드마크 인덱스 쌍 정의 (시각화 목적)
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # 엄지손가락 마디 연결선
    (0, 5), (5, 6), (6, 7), (7, 8),        # 검지손가락 마디 연결선
    (5, 9), (9, 10), (10, 11), (11, 12),   # 중지손가락 마디 연결선
    (9, 13), (13, 14), (14, 15), (15, 16), # 약지손가락 마디 연결선
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20) # 새끼손가락 마디 연결선
]

# ==============================================================================
# 2. 손지형(가위, 바위, 보) 판별 함수
# ==============================================================================
# 21개 관절 좌표(landmarks)를 전달받아 가위/바위/보 또는 알 수 없음 문자열을 반환하는 함수
def classify_rps(landmarks):
    # 검지 뿌리(5)와 새끼 뿌리(17)의 x좌표 위치를 비교하여 손바닥이 바라보는 방향 판별
    # 검지 뿌리가 오른쪽에 있으면 오른손, 왼쪽에 있으면 왼손 기준 적용
    if landmarks[5].x > landmarks[17].x:
        # 오른손 기준: 엄지 끝(4)의 x좌표가 엄지 마디(3)보다 오른쪽(더 큼)에 있으면 펴진 상태
        thumb_open = landmarks[4].x > landmarks[3].x
    else:
        # 왼손 기준: 엄지 끝(4)의 x좌표가 엄지 마디(3)보다 왼쪽(더 작음)에 있으면 펴진 상태
        thumb_open = landmarks[4].x < landmarks[3].x

    # 나머지 4개 손가락은 y축 좌표 비교 (화면 상단이 0, 하단이 1이므로 y값이 작을수록 위로 펴짐)
    # 검지 손가락 끝(8)이 중간 마디(6)보다 위(y값이 작음)에 있는지 확인
    index_open = landmarks[8].y < landmarks[6].y
    
    # 중지 손가락 끝(12)이 중간 마디(10)보다 위에 있는지 확인
    middle_open = landmarks[12].y < landmarks[10].y
    
    # 약지 손가락 끝(16)이 중간 마디(14)보다 위에 있는지 확인
    ring_open = landmarks[16].y < landmarks[14].y
    
    # 새끼 손가락 끝(20)이 중간 마디(18)보다 위에 있는지 확인
    pinky_open = landmarks[20].y < landmarks[18].y

    # 가위 조건 1: (엄지+검지)가 펴지고 중지, 약지, 새끼는 굽혀진 경우 (권총 모양)
    scissors_thumb_index = thumb_open and index_open and (not middle_open) and (not ring_open) and (not pinky_open)
    
    # 가위 조건 2: (검지+중지)가 펴지고 엄지, 약지, 새끼는 굽혀진 경우 (V 자 모양)
    scissors_index_middle = (not thumb_open) and index_open and middle_open and (not ring_open) and (not pinky_open)

    # 가위 조건 1 또는 2 중 하나라도 만족 시 "가위" 반환
    if scissors_thumb_index or scissors_index_middle:
        return "가위"
    
    # 보 조건: 엄지를 포함한 5개 손가락이 모두 펴진 경우 "보" 반환
    elif thumb_open and index_open and middle_open and ring_open and pinky_open:
        return "보"
    
    # 바위 조건: 엄지를 포함한 5개 손가락이 모두 굽혀진 경우 "바위" 반환
    elif (not thumb_open) and (not index_open) and (not middle_open) and (not ring_open) and (not pinky_open):
        return "바위"
    
    # 위 조건들에 해당하지 않는 부정확한 손짓일 경우 "알 수 없음" 반환
    else:
        return "알 수 없음"

# ==============================================================================
# 3. 메인 가위바위보 실행 및 UI 설정
# ==============================================================================
# MediaPipe HandLandmarker 모델 파라미터 및 인스턴스 옵션 구성
options = HandLandmarkerOptions(
    # 로컬 경로에 저장된 손 인식 모델 파일(.task) 지정
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    # 웹캠 비디오 프레임 스트림 처리를 위한 VIDEO 모드 선택
    running_mode=VisionRunningMode.VIDEO,
    # 추론할 최대 손 개수 (1개로 제한하여 오작동 방지)
    num_hands=1,
    # 최초 손 탐지 신뢰도 임계값 (50% 이상)
    min_hand_detection_confidence=0.5,
    # 프레임 간 손 추적 신뢰도 임계값 (50% 이상)
    min_tracking_confidence=0.5
)

# 0번 인덱스의 기본 웹캠 디바이스 연결
cap = cv2.VideoCapture(0)

# 가위바위보 게임에서 사용할 무작위 선택 리스트
choices = ["가위", "바위", "보"]

# 컴퓨터가 선택한 낸 값 저장 변수 초기화
computer_choice = None

# 플레이어가 선택한 낸 값 저장 변수 초기화
user_choice = None

# 화면에 표시될 게임 결과 텍스트 초기화
game_result = "Space키를 눌러 가위바위보 시작!"

# 사용자 누적 점수 변수 초기화
score_user = 0

# 컴퓨터 누적 점수 변수 초기화
score_computer = 0

# HandLandmarker 인스턴스를 안전하게 생성하고 자원을 관리하는 with 블록 시작
with HandLandmarker.create_from_options(options) as landmarker:
    # 카메라 장치가 정상적으로 열려 있는 동안 지속 반복
    while cap.isOpened():
        # 카메라로부터 프레임 1개를 읽어옴 (ret: 성공여부, frame: BGR 이미지)
        ret, frame = cap.read()
        
        # 프레임을 읽어오지 못했으면(카메라 연결 끊김 등) 반복문 탈출
        if not ret:
            break

        # 사용자가 거울처럼 보게 하기 위해 좌우 반전 처리
        frame = cv2.flip(frame, 1)
        
        # 화면에 관절을 그리기 위해 프레임의 높이(h)와 너비(w) 추출
        h, w, _ = frame.shape
        
        # OpenCV의 BGR 색상 채널을 MediaPipe용 RGB 색상 채널로 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # NumPy 이미지 배열을 MediaPipe 전용 Image 객체로 전환
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # VIDEO 모드에 필수적으로 필요한 밀리초(ms) 단위의 타임스탬프 계산
        frame_timestamp_ms = int(time.time() * 1000)
        
        # MediaPipe 모델에 프레임과 타임스탬프를 전달하여 손 관절 추론 수행
        result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        # 손 감지가 되지 않았을 때의 기본 상태 텍스트
        current_hand_gesture = "손을 올려주세요"

        # 손 관절 추론 결과가 존재할 경우
        if result.hand_landmarks:
            # 감지된 손 개수만큼 반복 (num_hands=1이므로 1회 실행)
            for hand_landmarks in result.hand_landmarks:
                # 21개 랜드마크의 화면 픽셀 좌표를 담을 리스트
                pixel_coords = []
                
                # 랜드마크 21개 좌표를 순회하며 픽셀 위치 계산
                for lm in hand_landmarks:
                    # 정규화된 0.0~1.0 좌표를 실제 화면 해상도 픽셀 좌표로 변환
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    pixel_coords.append((cx, cy))
                    
                    # 관절 위치에 초록색 원 그리기
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

                # 연결선 정의 배열(HAND_CONNECTIONS)에 따라 파란색 뼈대 그리기
                for start_idx, end_idx in HAND_CONNECTIONS:
                    cv2.line(frame, pixel_coords[start_idx], pixel_coords[end_idx], (255, 0, 0), 2)

                # 현재 캡처된 손 좌표를 분류 함수로 전달해 가위/바위/보 판단
                current_hand_gesture = classify_rps(hand_landmarks)

        # ======================================================================
        # 4. 게임 화면 UI 및 오버레이 텍스트 출력
        # ======================================================================
        # 상단 누적 스코어판 출력 (흰색)
        cv2.putText(frame, f"Score - You: {score_user} | Com: {score_computer}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        # 현재 카메라에 인식 중인 손 모양 텍스트 출력 (노란색)
        cv2.putText(frame, f"Detected Hand: {current_hand_gesture}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        # 승패 및 안내 문구 출력 (초록색)
        cv2.putText(frame, f"Result: {game_result}", (20, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # 컴퓨터의 선택이 존재할 경우 컴퓨터 선택값 출력 (연보라색)
        if computer_choice:
            cv2.putText(frame, f"Com Choice: {computer_choice}", (20, 170),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 250), 2)

        # 하단 조작법 안내 문구 출력 (회색)
        cv2.putText(frame, "Press 'Space' to Play / 'q' or ESC to Exit", (20, h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 1)

        # 완성된 이미지 프레임을 창에 표시
        cv2.imshow("Rock Paper Scissors Game", frame)

        # 5밀리초 동안 키보드 입력 대기
        key = cv2.waitKey(5) & 0xFF
        
        # 스페이스바(ASCII 32) 입력 시 가위바위보 승패 판정 실행
        if key == 32:
            # 인식된 손 모양이 가위, 바위, 보 중 하나일 때만 게임 진행
            if current_hand_gesture in choices:
                # 사용자의 현재 손 모양 저장
                user_choice = current_hand_gesture
                
                # 컴퓨터의 선택 무작위 추출
                computer_choice = random.choice(choices)

                # 사용자 선택과 컴퓨터 선택이 같은 경우 (무승부)
                if user_choice == computer_choice:
                    game_result = f"Draw! (You: {user_choice} vs Com: {computer_choice})"
                
                # 사용자가 이기는 경우 (가위>보, 바위>가위, 보>바위)
                elif (user_choice == "가위" and computer_choice == "보") or \
                     (user_choice == "바위" and computer_choice == "가위") or \
                     (user_choice == "보" and computer_choice == "바위"):
                    game_result = f"You Win! (You: {user_choice} vs Com: {computer_choice})"
                    score_user += 1 # 사용자 점수 1점 추가
                
                # 컴퓨터가 이기는 경우
                else:
                    game_result = f"You Lose! (You: {user_choice} vs Com: {computer_choice})"
                    score_computer += 1 # 컴퓨터 점수 1점 추가
            
            # 인식 상태가 "알 수 없음"이거나 손을 올리지 않은 경우
            else:
                game_result = "Hand position unclear! Try again."

        # 'q' 키 또는 ESC(ASCII 27) 키 입력 시 게임 루프 종료
        elif key == 27 or key == ord('q'):
            break

# ==============================================================================
# 5. 메인 루프 종료 후 자원 해제
# ==============================================================================
# OpenCv 카메라 비디오 스트림 연결 해제
cap.release()

# 모든 OpenCV 그래픽 창 닫기
cv2.destroyAllWindows()

In [ ]:

# OpenCV 라이브러리: 비디오 프레임 처리 및 화면 출력을 담당
import cv2

# MediaPipe 라이브러리: 손 관절(Landmark) 추론 및 컴퓨터 비전 기능 제공
import mediapipe as mp

# time 라이브러리: 1초 대기 및 쿨타임 시간 측정을 위해 사용
import time

# ==============================================================================
# 1. MediaPipe Tasks API 필수 모듈 및 설정 로드
# ==============================================================================
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # 엄지손가락 마디 연결선
    (0, 5), (5, 6), (6, 7), (7, 8),        # 검지손가락 마디 연결선
    (5, 9), (9, 10), (10, 11), (11, 12),   # 중지손가락 마디 연결선
    (9, 13), (13, 14), (14, 15), (15, 16), # 약지손가락 마디 연결선
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20) # 새끼손가락 마디 연결선
]

# ==============================================================================
# 2. 손지형(가위, 바위, 보) 판별 함수
# ==============================================================================
def classify_rps(landmarks):
    if landmarks[5].x > landmarks[17].x:
        thumb_open = landmarks[4].x > landmarks[3].x
    else:
        thumb_open = landmarks[4].x < landmarks[3].x

    index_open = landmarks[8].y < landmarks[6].y
    middle_open = landmarks[12].y < landmarks[10].y
    ring_open = landmarks[16].y < landmarks[14].y
    pinky_open = landmarks[20].y < landmarks[18].y

    scissors_thumb_index = thumb_open and index_open and (not middle_open) and (not ring_open) and (not pinky_open)
    scissors_index_middle = (not thumb_open) and index_open and middle_open and (not ring_open) and (not pinky_open)
    scissors_index_middle2 = (not thumb_open) and (not index_open) and (not middle_open) and ring_open and pinky_open

    if scissors_thumb_index or scissors_index_middle or scissors_index_middle2:
        return "가위"
    elif thumb_open and index_open and middle_open and ring_open and pinky_open:
        return "보"
    elif (not thumb_open) and (not index_open) and (not middle_open) and (not ring_open) and (not pinky_open):
        return "바위"
    else:
        return "Unknown"

# ==============================================================================
# 3. 메인 가위바위보 실행 및 UI 설정
# ==============================================================================
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0)

choices = ["가위", "바위", "보"]

score_left = 0
score_right = 0
game_result = "Show 2 hands to start!"

# 타이머 및 상태 관련 변수
both_hands_start_time = None  # 두 손이 동시에 감지되기 시작한 시각
last_game_time = 0            # 마지막으로 게임 결과가 나온 시각 (쿨타임 제어)
COOLDOWN_TIME = 3.0           # 판정 후 다음 게임까지 대기하는 시간 (초)

with HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        current_time = time.time()
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        frame_timestamp_ms = int(current_time * 1000)
        
        result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        detected_gestures = {"Left": "None", "Right": "None"}

        if result.hand_landmarks and result.handedness:
            for idx, hand_landmarks in enumerate(result.hand_landmarks):
                hand_label = result.handedness[idx][0].category_name
                
                pixel_coords = []
                for lm in hand_landmarks:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    pixel_coords.append((cx, cy))
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

                for start_idx, end_idx in HAND_CONNECTIONS:
                    cv2.line(frame, pixel_coords[start_idx], pixel_coords[end_idx], (255, 0, 0), 2)

                gesture = classify_rps(hand_landmarks)
                detected_gestures[hand_label] = gesture

                wrist_x, wrist_y = pixel_coords[0]
                cv2.putText(frame, f"{hand_label}: {gesture}", (wrist_x - 30, wrist_y + 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

        # ======================================================================
        # 4. 자동 타이머 및 게임 승패 로직
        # ======================================================================
        left_choice = detected_gestures["Left"]
        right_choice = detected_gestures["Right"]

        # 두 손 모두 정상 인식(가위/바위/보 중 하나) 되었고, 쿨타임이 지난 경우
        if left_choice in choices and right_choice in choices:
            if current_time - last_game_time > COOLDOWN_TIME:
                if both_hands_start_time is None:
                    both_hands_start_time = current_time  # 타이머 측정 시작
                
                elapsed_time = current_time - both_hands_start_time
                
                # 1초 유지 시 승패 판정 실행
                if elapsed_time >= 1.0:
                    if left_choice == right_choice:
                        game_result = f"Draw! ({left_choice} vs {right_choice})"
                    elif (user_choice == "가위" and computer_choice == "보") or \
                     (user_choice == "바위" and computer_choice == "가위") or \
                     (user_choice == "보" and computer_choice == "바위"):
                        game_result = f"Left Win! ({left_choice} vs {right_choice})"
                        score_left += 1
                    else:
                        game_result = f"Right Win! ({left_choice} vs {right_choice})"
                        score_right += 1

                    # 상태 리셋 및 쿨타임 갱신
                    both_hands_start_time = None
                    last_game_time = current_time
                else:
                    # 1초 대기 중 카운트다운 표시
                    game_result = f"Hold position... {1.0 - elapsed_time:.1f}s"
            else:
                # 판정 직후 쿨타임 동안 대기
                game_result = f"Next round in {COOLDOWN_TIME - (current_time - last_game_time):.1f}s"
        else:
            # 손이 인식 범위에서 벗어나거나 자세가 흐트러지면 타이머 초기화
            both_hands_start_time = None
            if current_time - last_game_time > COOLDOWN_TIME:
                game_result = "Show BOTH hands clearly!"

        # ======================================================================
        # 5. UI 오버레이 텍스트 출력
        # ======================================================================
        cv2.putText(frame, f"Score - Left: {score_left} | Right: {score_right}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        cv2.putText(frame, f"Left: {left_choice} | Right: {right_choice}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        cv2.putText(frame, f"Result: {game_result}", (20, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        cv2.putText(frame, "Press 'q' or ESC to Exit", (20, h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)

        cv2.imshow("Auto 2-Hand Rock Paper Scissors", frame)

        key = cv2.waitKey(5) & 0xFF
        if key == 27 or key == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import pyautogui

# 화면 크기 출력 (예: 1920x1080)
width, height = pyautogui.size()

# 현재 마우스 커서 위치 가져오기
x, y = pyautogui.position()

# 마우스 이동 (X: 100, Y: 200 위치로 1초 동안 이동)
pyautogui.moveTo(100, 200, duration=1.0)

# 클릭 제어
pyautogui.click()                      # 좌클릭
pyautogui.doubleClick()                # 더블 클릭
pyautogui.rightClick()                 # 우클릭
pyautogui.click(x=500, y=500)          # 특정 좌표 클릭

# 스크롤 (양수는 위로, 음수는 아래로)
pyautogui.scroll(-500)

In [12]:
# OpenCV 라이브러리: 비디오 프레임 캡처, 좌표 변환, 화면 시각화 담당
import cv2

# MediaPipe 라이브러리: 손 관절(21개 랜드마크) 추론을 담당
import mediapipe as mp

# PyAutoGUI 라이브러리: 파이썬으로 진짜 마우스 이동 및 클릭 제어
import pyautogui

# time 라이브러리: MediaPipe Tasks API에 넘겨줄 타임스탬프 생성용
import time

# math 라이브러리: 엄지와 검지 손가락 끝 사이의 유클리드 거리를 계산용
import math

# ==============================================================================
# 1. PyAutoGUI 환경 설정 및 모니터 해상도 구하기
# ==============================================================================
# 마우스 커서가 화면 모서리에 닿았을 때 발생할 수 있는 파이썬 예외 강제 종료 방지
pyautogui.FAILSAFE = False

# 마우스 동작 간 기본 대기 시간을 0.01초로 짧게 설정하여 커서 반응 속도 향상
pyautogui.PAUSE = 0.01

# 현재 사용 중인 모니터 화면의 전체 가로(screen_w), 세로(screen_h) 해상도 획득
screen_w, screen_h = pyautogui.size()

# ==============================================================================
# 2. MediaPipe Tasks API 설정 및 모델 로드
# ==============================================================================
# BaseOptions: .task 모델 파일 경로 지정용 클래스
BaseOptions = mp.tasks.BaseOptions

# HandLandmarker: 실제 손 관절 좌표 추론 클래스
HandLandmarker = mp.tasks.vision.HandLandmarker

# HandLandmarkerOptions: 신뢰도, 추론 모드, 손 개수 등 세부 옵션 지정 클래스
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

# VisionRunningMode: 비디오 프레임 스트림 방식 선택용 클래스
VisionRunningMode = mp.tasks.vision.RunningMode

# 손 관절 마디 연결 정보 정의 (시각화 목적)
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # 엄지손가락
    (0, 5), (5, 6), (6, 7), (7, 8),        # 검지손가락
    (5, 9), (9, 10), (10, 11), (11, 12),   # 중지손가락
    (9, 13), (13, 14), (14, 15), (15, 16), # 약지손가락
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20) # 새끼손가락
]

# HandLandmarker 모델 세부 옵션 지정
options = HandLandmarkerOptions(
    # 로컬 경로에 있는 모델(.task) 지정
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    # 실시간 비디오 스트림 처리 모드로 설정
    running_mode=VisionRunningMode.VIDEO,
    # 최대 인식할 손 개수 (마우스 제어이므로 1개로 제한)
    num_hands=1,
    # 최초 손 감지 신뢰도 임계값
    min_hand_detection_confidence=0.5,
    # 지속 손 추적 신뢰도 임계값
    min_tracking_confidence=0.5
)

# 기본 웹캠 캡처 객체 생성 (0번 카메라)
cap = cv2.VideoCapture(0)

# 이전 프레임의 마우스 커서 위치 기록용 변수 (부드러운 보정 계산용)
prev_x, prev_y = 0, 0

# 마우스 커서의 떨림 현상을 완화하기 위한 보정 비율 (값이 클수록 부드럽지만 약간의 딜레이 발생)
smoothing = 5

# 클릭 상태를 유지/해제하여 프레임마다 연타로 클릭되는 현상을 방지하는 플래그
is_clicked = False

# 카메라 가장자리 여백 영역 설정 (손을 웹캠 끝까지 움직이지 않아도 모니터 구석에 닿도록 함)
margin = 100 

# ==============================================================================
# 3. 실시간 가상 마우스 루프 시작
# ==============================================================================
# Landmarker 인스턴스 자동 자원 관리를 위한 with 블록 시작
with HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        # 웹캠 프레임 가져오기 (ret: 성공 여부, frame: BGR 이미지)
        ret, frame = cap.read()
        if not ret:
            break

        # 사용자가 거울 보듯 자연스럽게 조작하도록 좌우 반전
        frame = cv2.flip(frame, 1)
        
        # 현재 카메라 프레임의 높이(h), 너비(w) 추출
        h, w, _ = frame.shape
        
        # OpenCV의 BGR 이미지를 MediaPipe용 RGB 이미지로 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # NumPy 이미지 배열을 MediaPipe 전용 Image 객체로 래핑
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # VIDEO 모드 필수 요구사항: 밀리초(ms) 단위의 단조 증가 타임스탬프 계산
        frame_timestamp_ms = int(time.time() * 1000)
        
        # MediaPipe 모델로 손 좌표 추론 실행
        result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        # 카메라 화면 상에 노란색 가상 조작 영역 Box 표시
        cv2.rectangle(frame, (margin, margin), (w - margin, h - margin), (255, 255, 0), 2)

        # 손 관절 추론 결과가 존재하는 경우
        if result.hand_landmarks:
            for hand_landmarks in result.hand_landmarks:
                # 21개 관절의 픽셀 좌표를 보관할 리스트
                pixel_coords = []
                
                # 랜드마크 정규화 좌표(0.0~1.0)를 해상도에 맞는 픽셀 좌표로 환산하여 시각화
                for lm in hand_landmarks:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    pixel_coords.append((cx, cy))
                    # 관절 지점에 초록색 점 그리기
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

                # 손가락 관절 간 파란색 선으로 연결선 그리기
                for start_idx, end_idx in HAND_CONNECTIONS:
                    cv2.line(frame, pixel_coords[start_idx], pixel_coords[end_idx], (255, 0, 0), 2)

                # 마우스 커서 조작 및 클릭 판정에 사용할 엄지 끝(4번)과 검지 끝(8번) 좌표 추출
                thumb_x, thumb_y = pixel_coords[4]
                index_x, index_y = pixel_coords[8]

                # --------------------------------------------------------------
                # 3-1. 마우스 이동 처리 (검지 손가락 끝 좌표 기준)
                # --------------------------------------------------------------
                # 카메라 조작 영역(margin 적용 영역) 좌표를 실제 모니터 해상도 좌표 비율로 선형 매핑
                target_x = (index_x - margin) / (w - 2 * margin) * screen_w
                target_y = (index_y - margin) / (h - 2 * margin) * screen_h

                # 모니터 화면 바깥으로 커서가 벗어나지 않도록 좌표값 상하한 제한
                target_x = max(0, min(screen_w, target_x))
                target_y = max(0, min(screen_h, target_y))

                # 이동 보정 알고리즘: 이전 좌표와 목표 좌표 사이를 보정하여 손떨림으로 인한 마우스 튐 방지
                curr_x = prev_x + (target_x - prev_x) / smoothing
                curr_y = prev_y + (target_y - prev_y) / smoothing

                # PyAutoGUI를 통해 실제 OS 마우스 위치 이동
                pyautogui.moveTo(curr_x, curr_y)
                
                # 다음 프레임 보정을 위한 현재 위치 저장
                prev_x, prev_y = curr_x, curr_y

                # --------------------------------------------------------------
                # 3-2. 마우스 좌클릭 처리 (검지와 엄지 손가락 끝 간 거리 계산)
                # --------------------------------------------------------------
                # 2차원 피타고라스 정리(유클리드 거리)로 두 손가락 끝의 픽셀 거리 계산
                distance = math.hypot(index_x - thumb_x, index_y - thumb_y)
                
                # 엄지와 검지 사이를 연결하는 노란색 직선 그리기 (집게 동작 시각화)
                cv2.line(frame, (thumb_x, thumb_y), (index_x, index_y), (0, 255, 255), 2)

                # 거리가 30픽셀 미만으로 좁혀지면(손가락 맞닿음/집게 동작) 클릭 인식
                if distance < 30:
                    # 클릭 동작 인식 시 검지 끝에 빨간색 동그라미 표시
                    cv2.circle(frame, (index_x, index_y), 10, (0, 0, 255), -1)
                    
                    # 단발성 클릭 처리를 위해 is_clicked 플래그 사용
                    if not is_clicked:
                        pyautogui.click() # 좌클릭 실행
                        is_clicked = True  # 클릭 중 상태로 전환
                else:
                    # 손가락을 떼면 클릭 상태 해제
                    is_clicked = False

        # 가상 마우스 조작 화면 출력
        cv2.imshow("Virtual Mouse", frame)

        # 5밀리초 동안 키 입력 대기 ('q' 키 또는 ESC 입력 시 반복 탈출)
        key = cv2.waitKey(5) & 0xFF
        if key == 27 or key == ord('q'):
            break

# ==============================================================================
# 4. 자원 해제
# ==============================================================================
# 웹캠 장치 연결 해제
cap.release()

# 모든 OpenCV 그래픽 창 닫기
cv2.destroyAllWindows()

In [15]:
# OpenCV 라이브러리: 카메라 영상 처리 및 화면 시각화 담당
import cv2

# MediaPipe 라이브러리: 얼굴 랜드마크 추론 모듈 제공
import mediapipe as mp

# time 라이브러리: MediaPipe Video 모드에 필요한 타임스탬프 계산용
import time

# ==============================================================================
# 1. MediaPipe Tasks API 필수 모듈 및 설정 로드
# ==============================================================================
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Face Landmarker 세부 설정
options = FaceLandmarkerOptions(
    # 모델 파일 경로 지정 (.task 파일)
    base_options=BaseOptions(model_asset_path='face_landmarker.task'),
    # 실시간 비디오 스트림 처리 모드 선택
    running_mode=VisionRunningMode.VIDEO,
    # 감지할 최대 얼굴 개수
    num_faces=5,
    # 얼굴 감지 최저 신뢰도 (50% 이상)
    min_face_detection_confidence=0.5,
    # 프레임 간 추적 최저 신뢰도 (50% 이상)
    min_tracking_confidence=0.5
)

# 0번 인덱스 기본 웹캠 연결
cap = cv2.VideoCapture(0)

# ==============================================================================
# 2. 실시간 얼굴 특징점 시각화 루프
# ==============================================================================
with FaceLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 거울 모드를 위한 좌우 반전
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        # BGR 이미지를 RGB 이미지로 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # MediaPipe 전용 Image 객체로 래핑
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # 밀리초(ms) 단위 타임스탬프 계산
        frame_timestamp_ms = int(time.time() * 1000)

        # 얼굴 랜드마크 추론 실행
        result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        # 얼굴이 감지되었을 경우 특징점 그리기
        if result.face_landmarks:
            for face_landmarks in result.face_landmarks:
                # --------------------------------------------------------------
                # A. 468개 전체 입체 특징점(점) 찍기
                # --------------------------------------------------------------
                for lm in face_landmarks:
                    # 정규화된 좌표(0.0~1.0)를 실제 화면 픽셀 좌표로 환산
                    cx, cy = int(lm.x * w), int(lm.y * h)

                    # 관절 위치에 작은 초록색 원 그리기 (반지름: 1px)
                    cv2.circle(frame, (cx, cy), 1, (0, 255, 0), -1)

                # --------------------------------------------------------------
                # B. 주요 부위 강조 표시 (눈, 눈썹, 입술, 얼굴 윤곽 등)
                # --------------------------------------------------------------
                # 주요 랜드마크 인덱스 포인트 예시
                # 1번: 코 끝, 10번: 이마 중앙 상단, 152번: 턱 끝
                # 33번: 왼쪽 눈 가장자리, 263번: 오른쪽 눈 가장자리
                key_indices = [1, 10, 152, 33, 263, 61, 291]

                for idx in key_indices:
                    lm = face_landmarks[idx]
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    # 핵심 특징점은 빨간색 큰 원으로 강조
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

        # 화면 안내 문구 표시
        cv2.putText(frame, "Face Landmarks Visualization", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, "Press 'q' or ESC to exit", (20, h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 1)

        # 결과 프레임 출력
        cv2.imshow("Face Landmarks", frame)

        # 'q' 키 또는 ESC 키 입력 시 종료
        key = cv2.waitKey(5) & 0xFF
        if key == 27 or key == ord('q'):
            break

# ==============================================================================
# 3. 자원 해제
# ==============================================================================
# OpenCv 카메라 비디오 스트림 연결 해제
cap.release()

# 모든 OpenCV 그래픽 창 닫기
cv2.destroyAllWindows()

In [18]:
import cv2
import numpy as np

def create_mask_png(filename, text, bg_color):
    # 400x400 크기의 투명 배경(BGRA: 4채널) 이미지 생성
    # [B, G, R, Alpha] - Alpha 값이 0이면 완전 투명
    size = 400
    img = np.zeros((size, size, 4), dtype=np.uint8)

    # 1. 가면 형태의 채워진 원 그리기 (원 영역의 Alpha를 200~230으로 설정하여 반투명 효과)
    center = (size // 2, size // 2)
    radius = size // 2 - 20
    
    # BGR 색상에 Alpha(투명도 220) 채널 추가
    color_with_alpha = bg_color + (220,)
    cv2.circle(img, center, radius, color_with_alpha, -1)
    
    # 테두리 선 그리기 (완전 불투명 Alpha 255)
    border_color_with_alpha = (255, 255, 255, 255)
    cv2.circle(img, center, radius, border_color_with_alpha, 8)

    # 2. 가면 중앙에 글자(바위/가위/보) 쓰기
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 2.5
    thickness = 5
    
    # 텍스트 크기 계산하여 중앙 정렬
    text_size = cv2.getTextSize(text, font, font_scale, thickness)[0]
    text_x = (size - text_size[0]) // 2
    text_y = (size + text_size[1]) // 2

    # 흰색 글씨 쓰기 (Alpha 255)
    cv2.putText(img, text, (text_x, text_y), font, font_scale, (255, 255, 255, 255), thickness)

    # PNG 파일로 저장
    cv2.imwrite(filename, img)
    print(f"성공적으로 생성됨: {filename}")

# 1. 바위 마우스/가면 (파란색 계열)
create_mask_png('mask_rock.png', 'ROCK', (255, 100, 0))

# 2. 가위 마우스/가면 (빨간색 계열)
create_mask_png('mask_scissors.png', 'SCISSORS', (50, 50, 255))

# 3. 보 마우스/가면 (초록색 계열)
create_mask_png('mask_paper.png', 'PAPER', (50, 200, 50))

성공적으로 생성됨: mask_rock.png
성공적으로 생성됨: mask_scissors.png
성공적으로 생성됨: mask_paper.png


In [21]:
# OpenCV 라이브러리: 비디오 프레임 캡처, 이미지 가공, 텍스트/도형 시각화, 화면 출력 담당
import cv2

# MediaPipe 라이브러리: 인공지능 기반 손 관절 추론 및 얼굴 랜드마크 데이터 제공
import mediapipe as mp

# random 라이브러리: 컴퓨터의 가위/바위/보 선택을 무작위로 추출하기 위해 사용
import random

# time 라이브러리: MediaPipe Video 모드 실행 시 필요한 밀리초(ms) 단위 타임스탬프 계산용
import time

# numpy 라이브러리: 고성능 행렬 연산 및 이미지 배열 데이터 처리를 위한 모듈
import numpy as np

# ==============================================================================
# 1. MediaPipe Tasks API 모듈 로드 및 객체 정의
# ==============================================================================
# BaseOptions: .task 모델 파일의 위치 및 실행 디바이스(CPU/GPU) 환경 설정 모듈
BaseOptions = mp.tasks.BaseOptions

# HandLandmarker: 21개 손 관절 좌표 수집 및 추론 메인 클래스
HandLandmarker = mp.tasks.vision.HandLandmarker

# HandLandmarkerOptions: 손 탐지 임계값, 감지 손 개수, 비디오 모드 세팅 클래스
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

# FaceLandmarker: 468개 얼굴 이목구비 및 입체 Mesh 좌표 추론 메인 클래스
FaceLandmarker = mp.tasks.vision.FaceLandmarker

# FaceLandmarkerOptions: 얼굴 탐지 임계값, 인식 얼굴 개수, 비디오 모드 세팅 클래스
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions

# VisionRunningMode: 처리 모드 설정 (LIVE_STREAM, VIDEO, IMAGE 중 VIDEO 선택)
VisionRunningMode = mp.tasks.vision.RunningMode

# 손 관절 뼈대를 시각적으로 이어줄 21개 랜드마크 인덱스 연결 쌍 정의
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # 엄지손가락 마디 연결선 (손목 -> 손가락 끝)
    (0, 5), (5, 6), (6, 7), (7, 8),        # 검지손가락 마디 연결선
    (5, 9), (9, 10), (10, 11), (11, 12),   # 중지손가락 마디 연결선
    (9, 13), (13, 14), (14, 15), (15, 16), # 약지손가락 마디 연결선
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20) # 새끼손가락 마디 연결선
]

# ==============================================================================
# 2. 투명 배경(Alpha 채널) PNG 가면 이미지 알파 블렌딩 합성 함수
# ==============================================================================
def overlay_transparent(background, overlay, x, y, size=None):
    """
    background: 배경이 될 카메라 프레임 (BGR 이미지)
    overlay: 씌울 가면 이미지 (4채널 BGRA 투명 PNG)
    x, y: 가면이 위치할 좌상단 픽셀 좌표
    size: 변경할 (너비, 높이) 크기 튜플
    """
    # 사용자의 얼굴 크기에 맞춰 가면 이미지 크기를 동적으로 조정
    if size and size[0] > 0 and size[1] > 0:
        overlay = cv2.resize(overlay, size, interpolation=cv2.INTER_AREA)

    # 가면 이미지의 높이, 너비, 채널 수 추출
    h, w, c = overlay.shape
    bg_h, bg_w, _ = background.shape

    # 가면이 카메라 화면 바깥으로 나갈 경우 잘려나갈 경계 범위 계산 (오류 방지)
    bg_y1, bg_y2 = max(0, y), min(bg_h, y + h)
    bg_x1, bg_x2 = max(0, x), min(bg_w, x + w)

    overlay_y1, overlay_y2 = max(0, -y), h - max(0, (y + h) - bg_h)
    overlay_x1, overlay_x2 = max(0, -x), w - max(0, (x + w) - bg_w)

    # 가면이 화면 영역 전체를 벗어났다면 원본 이미지 그대로 반환
    if overlay_y1 >= overlay_y2 or overlay_x1 >= overlay_x2:
        return background

    # 4번째 채널인 Alpha(투명도: 0~255) 값을 0.0 ~ 1.0 비율 범위로 정규화
    alpha = overlay[overlay_y1:overlay_y2, overlay_x1:overlay_x2, 3] / 255.0

    # B, G, R 3개 색상 채널별로 투명 합성 연산 수행 (알파 블렌딩 공식)
    for color in range(3):
        background[bg_y1:bg_y2, bg_x1:bg_x2, color] = (
            alpha * overlay[overlay_y1:overlay_y2, overlay_x1:overlay_x2, color] +
            (1.0 - alpha) * background[bg_y1:bg_y2, bg_x1:bg_x2, color]
        )
    return background

# ==============================================================================
# 3. 손 관절 좌표 기반 가위, 바위, 보 동작 판별 함수
# ==============================================================================
def classify_rps(landmarks):
    """
    landmarks: MediaPipe 손 관절 21개 좌표 리스트
    반환값: "가위", "바위", "보", "알 수 없음" 문자열
    """
    # 검지 뿌리(5)와 새끼 뿌리(17)의 x좌표를 비교하여 손바닥이 앞/뒤 중 어디를 보는지 확인
    if landmarks[5].x > landmarks[17].x:
        # 오른손/손바닥이 보일 때: 엄지 끝(4)이 엄지 마디(3)보다 우측(x가 더 큼)이면 펴진 상태
        thumb_open = landmarks[4].x > landmarks[3].x
    else:
        # 왼손/손등이 보일 때: 엄지 끝(4)이 엄지 마디(3)보다 좌측(x가 더 작음)이면 펴진 상태
        thumb_open = landmarks[4].x < landmarks[3].x

    # 나머지 손가락들은 y축 좌표 비교 (화면 상단이 0, 하단이 1이므로 y가 작을수록 펴진 것)
    index_open = landmarks[8].y < landmarks[6].y    # 검지 끝(8)이 관절(6)보다 위인지 확인
    middle_open = landmarks[12].y < landmarks[10].y  # 중지 끝(12)이 관절(10)보다 위인지 확인
    ring_open = landmarks[16].y < landmarks[14].y    # 약지 끝(16)이 관절(14)보다 위인지 확인
    pinky_open = landmarks[20].y < landmarks[18].y   # 새끼 끝(20)이 관절(18)보다 위인지 확인

    # 가위 조건 1: 엄지와 검지가 펴지고 나머지 굽힘 (권총 모양 가위)
    scissors_thumb_index = thumb_open and index_open and (not middle_open) and (not ring_open) and (not pinky_open)
    # 가위 조건 2: 검지와 중지가 펴지고 나머지 굽힘 (V 자 모양 가위)
    scissors_index_middle = (not thumb_open) and index_open and middle_open and (not ring_open) and (not pinky_open)

    # 가위 조건 1 또는 2 중 하나라도 만족하면 "가위"
    if scissors_thumb_index or scissors_index_middle:
        return "가위"
    # 모든 손가락 5개가 펴져 있다면 "보"
    elif thumb_open and index_open and middle_open and ring_open and pinky_open:
        return "보"
    # 모든 손가락 5개가 굽혀져 있다면 "바위"
    elif (not thumb_open) and (not index_open) and (not middle_open) and (not ring_open) and (not pinky_open):
        return "바위"
    # 규정되지 않은 손모양일 경우 "알 수 없음"
    else:
        return "알 수 없음"

# ==============================================================================
# 4. 가면 이미지 로드 및 MediaPipe 모델 옵션 초기화
# ==============================================================================
# 동작 판별 텍스트와 매핑되는 마스크 PNG 파일 불러오기 (4채널 IMREAD_UNCHANGED 옵션 필수)
masks = {
    "바위": cv2.imread('mask_rock.png', cv2.IMREAD_UNCHANGED),
    "가위": cv2.imread('mask_scissors.png', cv2.IMREAD_UNCHANGED),
    "보": cv2.imread('mask_paper.png', cv2.IMREAD_UNCHANGED)
}

# 얼굴 랜드마크 모델 옵션 세팅
face_options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='face_landmarker.task'), # 모델 파일 지정
    running_mode=VisionRunningMode.VIDEO,                             # 실시간 비디오 스트림 처리 모드
    num_faces=1,                                                      # 탐지할 최대 얼굴 수 (1개)
    min_face_detection_confidence=0.5,                                # 얼굴 탐지 임계값 (50%)
    min_tracking_confidence=0.5                                       # 얼굴 추적 임계값 (50%)
)

# 손 관절 모델 옵션 세팅
hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'), # 모델 파일 지정
    running_mode=VisionRunningMode.VIDEO,                             # 실시간 비디오 스트림 처리 모드
    num_hands=1,                                                      # 탐지할 최대 손 수 (1개로 혼선 방지)
    min_hand_detection_confidence=0.5,                                # 손 탐지 임계값 (50%)
    min_tracking_confidence=0.5                                       # 손 추적 임계값 (50%)
)

# 기본 시스템 카메라(인덱스 0번 웹캠) 오픈
cap = cv2.VideoCapture(0)

# 가위바위보 게임 상태 관리 변수 세팅
choices = ["가위", "바위", "보"]   # 컴퓨터 무작위 추첨 후보군 리스트
computer_choice = None           # 컴퓨터가 선택한 가위/바위/보 저장 변수
user_choice = None               # 플레이어가 선택한 가위/바위/보 저장 변수
game_result = "Space키를 눌러 가위바위보 시작!" # 게임 진행 현황 안내 문구
score_user = 0                   # 사용자의 누적 점수
score_computer = 0               # 컴퓨터의 누적 점수

# ==============================================================================
# 5. 실시간 추론 및 종합 처리 메인 루프
# ==============================================================================
# FaceLandmarker와 HandLandmarker 자원을 동시에 안전하게 오픈
with FaceLandmarker.create_from_options(face_options) as face_landmarker, \
     HandLandmarker.create_from_options(hand_options) as hand_landmarker:
    
    # 웹캠 연결이 정상이면 계속 반복 실행
    while cap.isOpened():
        # 프레임 단위로 카메라 화면 읽기 (ret: 성공여부, frame: BGR 이미지)
        ret, frame = cap.read()
        if not ret:
            break

        # 사용자가 거울을 보는 것처럼 느낄 수 있도록 화면 좌우 반전
        frame = cv2.flip(frame, 1)
        
        # 픽셀 좌표 계산을 위한 현재 프레임의 높이(h)와 너비(w) 추출
        h, w, _ = frame.shape
        
        # OpenCV의 BGR 채널 포맷을 MediaPipe 표준 RGB 채널로 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # NumPy 이미지 배열을 MediaPipe 프레임워크 전용 Image 객체로 패킹
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # VIDEO 실행 모드 규격에 요구되는 밀리초(ms) 단위의 현재 시간 타임스탬프 계산
        frame_timestamp_ms = int(time.time() * 1000)
        
        # AI 모델에 이미지를 전달하여 얼굴 및 손 동작 추론 결과 동시 추출
        face_result = face_landmarker.detect_for_video(mp_image, frame_timestamp_ms)
        hand_result = hand_landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        # ----------------------------------------------------------------------
        # A. 손 관절 시각화 및 가위바위보 동작 분류
        # ----------------------------------------------------------------------
        # 손 감지가 되지 않았을 때의 기본 안내 문구
        current_hand_gesture = "손을 올려주세요"

        # 카메라에 손이 인식되어 결과 좌표가 존재하는 경우
        if hand_result.hand_landmarks:
            for hand_landmarks in hand_result.hand_landmarks:
                # 21개 관절의 실제 픽셀 좌표를 저장할 리스트
                pixel_coords = []
                
                # 21개 랜드마크를 순회하며 화면 상의 (x, y) 픽셀 위치로 변환
                for lm in hand_landmarks:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    pixel_coords.append((cx, cy))
                    # 관절 위치마다 노란색 원 그리기
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 255), -1)

                # HAND_CONNECTIONS 인덱스 연결 정보에 따라 파란색 뼈대 라인 그리기
                for start_idx, end_idx in HAND_CONNECTIONS:
                    cv2.line(frame, pixel_coords[start_idx], pixel_coords[end_idx], (255, 0, 0), 2)

                # 손 관절 좌표 데이터를 판별 함수에 전달하여 가위/바위/보 판단
                current_hand_gesture = classify_rps(hand_landmarks)

        # ----------------------------------------------------------------------
        # B. 얼굴 특징점 표현 및 마스크 이미지 동적 오버레이
        # ----------------------------------------------------------------------
        # 카메라에 얼굴이 인식되어 결과 좌표가 존재하는 경우
        if face_result.face_landmarks:
            for face_landmarks in face_result.face_landmarks:
                # 1. 얼굴 전체 468개 랜드마크에 초록색 미세 점 그리기
                for lm in face_landmarks:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(frame, (cx, cy), 1, (0, 255, 0), -1)

                # 2. 이목구비 핵심 관절 포인트(코 끝, 이마, 턱, 눈 외각, 입술 끝) 강조
                key_indices = [1, 10, 152, 33, 263, 61, 291]
                for idx in key_indices:
                    lm = face_landmarks[idx]
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    # 중요 포인트는 빨간색 큰 원으로 강조
                    cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

                # 3. 손 모양(가위/바위/보)에 해당하는 마스크가 정상 로드되어 있다면 오버레이 처리
                if current_hand_gesture in masks and masks[current_hand_gesture] is not None:
                    mask_img = masks[current_hand_gesture]

                    # 얼굴 윤곽의 사방 끝 관절 좌표 추출 (10: 이마 상단, 152: 턱 끝, 234: 왼쪽 광대, 454: 오른쪽 광대)
                    left_x = int(face_landmarks[234].x * w)
                    right_x = int(face_landmarks[454].x * w)
                    top_y = int(face_landmarks[10].y * h)
                    bottom_y = int(face_landmarks[152].y * h)

                    # 감지된 얼굴의 가로/세로 폭을 측정하고 약간의 여유분(1.3배) 추가
                    mask_w = int((right_x - left_x) * 1.3)
                    mask_h = int((bottom_y - top_y) * 1.3)

                    # 얼굴 중심점 위치 계산
                    center_x = (left_x + right_x) // 2
                    center_y = (top_y + bottom_y) // 2
                    
                    # 가면의 좌상단 시작 픽셀 좌표 계산
                    mask_x = center_x - (mask_w // 2)
                    mask_y = center_y - (mask_h // 2)

                    # 투명 배경 오버레이 함수를 호출하여 화면에 가면 합성
                    frame = overlay_transparent(frame, mask_img, mask_x, mask_y, (mask_w, mask_h))

        # ----------------------------------------------------------------------
        # C. 게임 화면 UI 및 정보 텍스트 오버레이
        # ----------------------------------------------------------------------
        # 상단 누적 스코어 현황 출력 (흰색 텍스트)
        cv2.putText(frame, f"Score - You: {score_user} | Com: {score_computer}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        # 현재 카메라에 감지 중인 손 모양 텍스트 출력 (노란색 텍스트)
        cv2.putText(frame, f"Detected Hand: {current_hand_gesture}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        # 가위바위보 대결 승패 결과 및 안내 문구 출력 (초록색 텍스트)
        cv2.putText(frame, f"Result: {game_result}", (20, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # 컴퓨터가 가위/바위/보 중 하나를 낸 상태라면 컴퓨터의 선택 출력 (연보라색 텍스트)
        if computer_choice:
            cv2.putText(frame, f"Com Choice: {computer_choice}", (20, 170),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 250), 2)

        # 화면 하단 조작 키 사용 방법 안내 텍스트 출력 (회색 텍스트)
        cv2.putText(frame, "Press 'Space' to Play / 'q' or ESC to Exit", (20, h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 1)

        # 가공 완료된最終 프레임을 화면 윈도우 창에 렌더링
        cv2.imshow("Face Mask & Rock Paper Scissors Game", frame)

        # 키보드 입력을 5ms 동안 대기
        key = cv2.waitKey(5) & 0xFF
        
        # 스페이스바(ASCII 번호 32)를 누르면 판정 연산 실행
        if key == 32:
            # 인식된 손 모양이 가위, 바위, 보 중 하나로 확실할 때만 게임 진행
            if current_hand_gesture in choices:
                # 플레이어 선택값 확정
                user_choice = current_hand_gesture
                # 컴퓨터 무작위 추첨
                computer_choice = random.choice(choices)

                # 비긴 경우 처리
                if user_choice == computer_choice:
                    game_result = f"Draw! (You: {user_choice} vs Com: {computer_choice})"
                
                # 사용자가 승리하는 조건 판별 (가위>보, 바위>가위, 보>바위)
                elif (user_choice == "가위" and computer_choice == "보") or \
                     (user_choice == "바위" and computer_choice == "가위") or \
                     (user_choice == "보" and computer_choice == "바위"):
                    game_result = f"You Win! (You: {user_choice} vs Com: {computer_choice})"
                    score_user += 1 # 플레이어 점수 +1
                
                # 컴퓨터가 승리한 경우 처리
                else:
                    game_result = f"You Lose! (You: {user_choice} vs Com: {computer_choice})"
                    score_computer += 1 # 컴퓨터 점수 +1
            
            # 손 모양이 부정확하거나 화면에 손이 없는 경우 예외 처리
            else:
                game_result = "Hand position unclear! Try again."

        # 'q' 키 또는 ESC 키(ASCII 번호 27) 누를 시 루프 종료
        elif key == 27 or key == ord('q'):
            break

# ==============================================================================
# 6. 자원 해제 및 종료 처리
# ==============================================================================
# 카메라 디바이스 연결 해제 및 리소스 반환
cap.release()

# 오픈된 모든 OpenCV GUI 창 닫기
cv2.destroyAllWindows()